# **Shopee Reviews Sentiment Analysis in Indonesian Language**

**Table of Contents**

1. Cleaning Data
2. Prepocessing: Tokenization, Normalization, Stemmization, Stopwords
3. Labelling
4. Sentiment Classification
5. TF-IDF Vectorization
6. Training Model
7. Joblib Model

In [47]:
!pip install Sastrawi

import pandas as pd
import re
import numpy as np
import random
import joblib

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory, StopWordRemover, ArrayDictionary
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE

In [76]:
df = pd.read_csv('/content/Shopee Review Dataset.csv', nrows=150) #dataset source: https://www.kaggle.com/datasets/herafajrin/shopee-review
df = df[['ulasan', 'user', 'tanggal']]

In [77]:
df = pd.read_csv('/content/Shopee Review Dataset.csv')
df = df[['ulasan', 'user', 'tanggal']].dropna().drop_duplicates(subset='ulasan')
print(f"Total data yang dimuat: {len(df)}")

Total data yang dimuat: 130


**1. Cleaning Data**

In [78]:
def clean_shopee_review_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'@[A-Za-z0-9_]+|#\w+|RT[\s]+|https?://\S+', '', text)
    text = re.sub(r'[^A-Za-z0-9 ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['ulasan'] = df['ulasan'].apply(clean_shopee_review_text)

**2. Preprocessing**

Tokenization

In [83]:
tokenized = df['ulasan'].fillna('').apply(lambda x: x.split())

Normalization

In [84]:
norm = {}
with open('/content/Shopee Review Dictionary.csv', 'r') as file:
    for pair in file.read().split("', '"):
        pair = pair.strip("'")
        if "': '" in pair: key, value = pair.split("': '", 1); norm[key] = value

def normalisasi(str_text):
  return ' '.join([norm.get(word, word) for word in str_text.split()])

df['ulasan'] = df['ulasan'].fillna('').str.lower().apply(normalisasi)
data = df

Stemmization

In [85]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_text(text):
  return ' '.join([stemmer.stem(w) for w in text.split()])

df['ulasan'] = df['ulasan'].apply(stemming_text)

Stopword

In [86]:
stop_words = StopWordRemoverFactory().get_stop_words()
stop_words.extend(["tidak"])
stop_words_remover_new = StopWordRemover(ArrayDictionary(stop_words))
df['ulasan'] = df['ulasan'].apply(lambda x: stop_words_remover_new.remove(x))

df.to_csv('cleaned_sentiment_data.csv', index=False)

**3. Labelling**

In [87]:
pos_words = ['bagus', 'puas', 'cepat', 'aman', 'original', 'murah', 'mantap', 'oke', 'rekomendasi', 'terima kasih', 'suka']
neg_words = ['kecewa', 'jelek', 'lambat', 'penipu', 'kurang', 'buruk', 'rugi', 'bohong', 'rusak', 'tidak sesuai']

def custom_labeling(text):
    text = str(text).lower()
    score = 0
    for w in pos_words: score += text.count(w)
    for w in neg_words: score -= text.count(w)
    return 'Positif' if score >= 0 else 'Negatif'

data['klasifikasi'] = data['ulasan'].apply(custom_labeling)

In [88]:
dataset = data[['ulasan', 'klasifikasi']].apply(tuple, axis=1).tolist()

set_positif = [n for n in dataset if n[1] == 'Positif']
set_negatif = [n for n in dataset if n[1] == 'Negatif']

set_positif = random.sample(set_positif, k=int(len(set_positif)/2))
set_negatif = random.sample(set_negatif, k=int(len(set_negatif)/2))

train_set = set_positif + set_negatif

**4. Sentiment Classification**

In [89]:
data['klasifikasi'] = data['ulasan'].apply(custom_labeling)

**5. TF-IDF Vectorization**

In [90]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(data['ulasan'].fillna(''))

le = LabelEncoder()
y = le.fit_transform(data['klasifikasi'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [91]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('mnb', MultinomialNB())
])

param_grid = {
    'tfidf__max_features': [100, 500, 1000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [1, 5, 10],
    'tfidf__max_df': [0.7, 0.9, 1.0],
    'mnb__alpha': [0.1, 0.5, 1.0, 1.5, 2.0]
}

grid_search_pipeline = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_pipeline.fit(data['ulasan'].fillna(''), y)

print("Best parameters for the TF-IDF Vectorizer and Multinomial Naive Bayes pipeline:", grid_search_pipeline.best_params_)
print("Best cross-validation accuracy:", grid_search_pipeline.best_score_)

Fitting 5 folds for each of 270 candidates, totalling 1350 fits
Best parameters for the TF-IDF Vectorizer and Multinomial Naive Bayes pipeline: {'mnb__alpha': 0.1, 'tfidf__max_df': 0.7, 'tfidf__max_features': 500, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2)}
Best cross-validation accuracy: 0.976923076923077


In [92]:
if 'optimized_tfidf_vectorizer' not in locals() or 'X' not in locals():
    best_params = grid_search_pipeline.best_params_
    best_tfidf_params = {
        'max_features': best_params['tfidf__max_features'],
        'ngram_range': best_params['tfidf__ngram_range'],
        'min_df': best_params['tfidf__min_df'],
        'max_df': best_params['tfidf__max_df']
    }

    optimized_tfidf_vectorizer = TfidfVectorizer(**best_tfidf_params)
    X = optimized_tfidf_vectorizer.fit_transform(data['ulasan'].fillna(''))

**6. Training Model**

In [93]:
best_params = grid_search_pipeline.best_params_
best_tfidf_params = {
    'max_features': best_params['tfidf__max_features'],
    'ngram_range': best_params['tfidf__ngram_range'],
    'min_df': best_params['tfidf__min_df'],
    'max_df': best_params['tfidf__max_df']
}
best_mnb_alpha = best_params['mnb__alpha']

optimized_tfidf_vectorizer = TfidfVectorizer(**best_tfidf_params)
X = optimized_tfidf_vectorizer.fit_transform(data['ulasan'].fillna(''))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sm = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train, y_train)

class_counts_resampled = np.bincount(y_train_resampled)
total_samples_resampled = len(y_train_resampled)
class_prior_resampled = class_counts_resampled / total_samples_resampled

optimized_mnb_model = MultinomialNB(alpha=best_mnb_alpha, class_prior=class_prior_resampled)
optimized_mnb_model.fit(X_train_resampled, y_train_resampled)

MultinomialNB(alpha=0.1, class_prior=array([0.5, 0.5]))

In [94]:
y_pred_optimized = optimized_mnb_model.predict(X_test)
class_names = le.inverse_transform([0, 1])

print("\nClassification Report (Optimized Model after SMOTE):\n", classification_report(y_test, y_pred_optimized, target_names=class_names, zero_division=0))
print("Accuracy (Optimized Model after SMOTE):", accuracy_score(y_test, y_pred_optimized))


Classification Report (Optimized Model after SMOTE):
               precision    recall  f1-score   support

     Negatif       1.00      0.50      0.67         2
     Positif       0.96      1.00      0.98        24

    accuracy                           0.96        26
   macro avg       0.98      0.75      0.82        26
weighted avg       0.96      0.96      0.96        26

Accuracy (Optimized Model after SMOTE): 0.9615384615384616


**7. Joblib Model**

In [95]:
joblib.dump(optimized_mnb_model, 'optimized_multinomial_nb_model.joblib')
joblib.dump(optimized_tfidf_vectorizer, 'optimized_tfidf_vectorizer.joblib')
joblib.dump(le, 'optimized_label_encoder.joblib')

['optimized_label_encoder.joblib']

In [97]:
def predict_sentiment(text_list):
    clean_input = [str(t) for t in text_list]
    tfidf_matrix = optimized_tfidf_vectorizer.transform(clean_input)
    prediction = optimized_mnb_model.predict(tfidf_matrix)
    return le.inverse_transform(prediction)

test_text = ["Shopee sangat membantu saya dalam berbelanja"]
print(f'Text: {test_text[0]}')
print(f'Sentiment Prediction: {predict_sentiment(test_text)[0]}')

Text: Shopee sangat membantu saya dalam berbelanja
Sentiment Prediction: Positif
